# 01 – Metrics & Baseline
**Phase 1:** ResNet-50 features + logistic regression + surface-form text matching.
Expected val R-precision ≈ 0.50.

In [ ]:
import sys, os, pathlib
sys.path.insert(0, "/kaggle/input/datasets/somyajitsaha/food-retrieval-src")

import numpy as np
import pandas as pd
import torch

from src.config import seed_everything
SEED = 42
seed_everything(SEED)

import torchvision, timm, sklearn
print({k: v.__version__ for k, v in dict(
    torch=torch, torchvision=torchvision,
    timm=timm, sklearn=sklearn, numpy=np).items()})
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

# Paths
DATA_DIR = pathlib.Path("/kaggle/input/competitions/match-the-dish-to-its-rizz")
OUT_DIR  = pathlib.Path("/kaggle/working")
FEAT_DIR = OUT_DIR / "features" / "resnet50_in1k"
FEAT_DIR.mkdir(parents=True, exist_ok=True)
(OUT_DIR / "submissions").mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DATA_DIR exists:", DATA_DIR.exists())

In [ ]:
# ── Run metric unit tests ────────────────────────────────────────────────────
import subprocess, sys
result = subprocess.run(
    [sys.executable, "-m", "pytest",
     "/kaggle/input/datasets/somyajitsaha/food-retrieval-src/tests/test_metrics.py",
     "-v", "--tb=short", "-q"],
    capture_output=True, text=True
)
print(result.stdout[-3000:] if len(result.stdout) > 3000 else result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("Metric tests FAILED")
print("✓ All metric tests passed")

In [ ]:
# ── Load CSVs ────────────────────────────────────────────────────────────────
from src.data import load_csvs, parse_val_ground_truth

dfs          = load_csvs(DATA_DIR)
train_df     = dfs["train"]
val_gallery  = dfs["val_gallery"]
val_queries  = dfs["val_queries"]
val_gt_df    = dfs["val_gt"]
test_gallery = dfs["test_gallery"]
test_queries = dfs["test_queries"]
meta         = dfs["category_metadata"]

val_gt       = parse_val_ground_truth(val_gt_df)
cat_id_list  = meta["category_id"].tolist()         # list of string IDs
cat_id2idx   = {c: i for i, c in enumerate(cat_id_list)}
K            = len(cat_id_list)
print(f"K={K}  train={len(train_df)}  val={len(val_gallery)}  test={len(test_gallery)}")

In [ ]:
# ── Text matcher (100% accurate, verified locally) ───────────────────────────
from src.text_matching import build_surface_form_dict, TextMatcher, evaluate_text_matching

surface_forms = build_surface_form_dict(meta)
matcher       = TextMatcher(surface_forms, meta_df=meta)

val_q_labelled = val_queries.merge(val_gt_df[["query_id","category_id"]], on="query_id")
print("Val text matching accuracy:")
evaluate_text_matching(matcher, val_q_labelled)

In [ ]:
# ── Extract ResNet-50 features ────────────────────────────────────────────────
from torch.utils.data import DataLoader
from src.data import GalleryDataset, get_transforms
from src.models import load_timm_model
from src.features import extract_image_features, save_features, load_features

MEAN     = (0.485, 0.456, 0.406)
STD      = (0.229, 0.224, 0.225)
IMG_SIZE = 224
BATCH    = 128

model = load_timm_model("resnet50.a1_in1k", pretrained=True, num_classes=0, device=DEVICE)
tf    = get_transforms("val", size=IMG_SIZE, mean=MEAN, std=STD)

splits = {
    "train": (train_df,    DATA_DIR / "train_gallery"),
    "val":   (val_gallery, DATA_DIR / "validation_gallery"),
    "test":  (test_gallery,DATA_DIR / "test_gallery"),
}

for split, (df, img_dir) in splits.items():
    cache = FEAT_DIR / f"{split}_image.npy"
    if cache.exists():
        print(f"  {split}: cache hit, skipping")
        continue
    ds     = GalleryDataset(df, img_dir, transform=tf)
    loader = DataLoader(ds, batch_size=BATCH, shuffle=False, num_workers=4, pin_memory=True)
    feats, ids = extract_image_features(model, loader, device=DEVICE)
    save_features(feats, ids, FEAT_DIR, split)

del model
torch.cuda.empty_cache()
print("Feature extraction done")

In [ ]:
# ── Logistic regression on frozen features ────────────────────────────────────
from sklearn.linear_model import LogisticRegression

X_train, train_ids = load_features(FEAT_DIR, "train", gallery_df=train_df)
X_val,   val_ids   = load_features(FEAT_DIR, "val",   gallery_df=val_gallery)
X_test,  test_ids  = load_features(FEAT_DIR, "test",  gallery_df=test_gallery)

id2cat = dict(zip(train_df["image_id"], train_df["category_id"]))
y_str  = [id2cat[i] for i in train_ids]   # string category IDs
# map to zero-based index
y_enc  = np.array([cat_id2idx[c] for c in y_str])

clf = LogisticRegression(C=1.0, max_iter=1000, solver="lbfgs",
                          multi_class="multinomial", random_state=42, n_jobs=-1)
clf.fit(X_train, y_enc)
print(f"Train acc: {clf.score(X_train, y_enc):.4f}")

In [ ]:
# ── Evaluate on validation ────────────────────────────────────────────────────
from src.metrics import evaluate, print_metrics

S_val = clf.predict_proba(X_val)   # (N_val, K)

pred_dict = {}
for _, row in val_queries.iterrows():
    cat_id, _, _ = matcher.predict(str(row["description"]))
    cat_idx = cat_id2idx.get(cat_id, 0)
    order   = np.argsort(-S_val[:, cat_idx], kind="stable")
    pred_dict[str(row["query_id"])] = [val_ids[j] for j in order[:20]]

metrics = evaluate(pred_dict, val_gt)
print("\n=== Validation metrics ===")
print_metrics(metrics)

In [ ]:
# ── Build + validate + save submission ────────────────────────────────────────
from src.submit import build_submission, validate_submission, save_submission
import datetime

S_test = clf.predict_proba(X_test)   # (N_test, K)

test_query_ids, test_cat_indices = [], []
for _, row in test_queries.iterrows():
    cat_id, _, _ = matcher.predict(str(row["description"]))
    test_query_ids.append(str(row["query_id"]))
    test_cat_indices.append(cat_id2idx.get(cat_id, 0))

sub_df   = build_submission(S_test, test_ids, test_query_ids, test_cat_indices, top=20)
is_valid = validate_submission(sub_df, test_gallery)

if is_valid:
    ts       = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    sub_path = OUT_DIR / "submissions" / f"{ts}_baseline_resnet50.csv"
    sub_df.to_csv(sub_path, index=False)
    # Kaggle reads from /kaggle/working/submission.csv
    sub_df.to_csv(OUT_DIR / "submission.csv", index=False)
    print(f"Saved: {sub_path}")

print(f"""
=== Experiment log row ===
| E001 | {datetime.date.today()} | 1 | ResNet-50 + LogReg + surface-form | C=1.0, seed=42 | 42 | {metrics['r_precision']:.4f} | {metrics['map@20']:.4f} | {metrics['p@5']:.4f} | {metrics['hit@1']:.4f} | {metrics['mrr@20']:.4f} | baseline |
""")

In [ ]:
# ── Official format validator ─────────────────────────────────────────────────
import subprocess, sys
result = subprocess.run(
    [sys.executable,
     str(DATA_DIR / "validate_submission.py"),
     str(OUT_DIR / "submission.csv"),
     "--data-dir", str(DATA_DIR)],
    capture_output=True, text=True
)
print(result.stdout)
if result.stderr: print(result.stderr)
print("Validator exit code:", result.returncode)